In [3]:
import os
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import json

load_dotenv()

llm_url = os.getenv("LLM_URL")
llm_model = os.getenv("LLM_MODEL")
api_key = os.getenv("API_KEY")
llm_provider = os.getenv("LLM_PROVIDER")

model = init_chat_model(
    model=llm_model,
    base_url=llm_url,
    api_key=api_key,
    temperature=0,
    timeout=60,
    max_retries=3,
    model_provider=llm_provider,
)

json_path = "../data/docile/annotations/b6a51fed333341f1b51586fb.json"


def obtain_query(model, json_path):

    with open(json_path, "r") as f:
        ground_truth = json.load(f)

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "Answer the user's query"),
            (
                "human",
                "Based on {ground_truth}, which is a perfect parsed table, develop a query that would be relevant for a company (for example: retrieve all transactions of type 'X' and select from them fields 'Y' and 'Z', ensuring you do not select all available fields). Then, build a pydantic model according to this query",
            ),
        ]
    )

    chain = prompt | model

    query = chain.invoke({"ground_truth": json.dumps(ground_truth)})

    return query.content

In [4]:
print(obtain_query(model, json_path))

Based on the provided JSON data, which represents a parsed invoice/order document, here is a relevant business query and the corresponding Pydantic model.

### 1. Business Query Scenario

**Context:** A finance analyst needs to reconcile the specific line items for a campaign to verify the "Spots per Week" and the "Gross Rate" applied, without needing the full billing address or document metadata.

**Query:**
> "Retrieve all line items for the order where the currency is USD. For each line item, select only the **line item position**, the **date**, the **quantity** (labeled as 'SPOTS/WK'), and the **gross unit price**. Exclude all other fields such as the customer name, total amounts, and document IDs."

**SQL-like Representation:**
```sql
SELECT 
    line_item_position, 
    line_item_date, 
    line_item_quantity, 
    line_item_unit_price_gross
FROM 
    order_line_items
WHERE 
    currency = 'USD'
    AND document_type = 'order';
```

---

### 2. Pydantic Model

This model is desig